# 11 — Robustness checks (issues #35, #37)

Since Oct 2 2026, the project's primary results are built from the **course's OMOP tables**, so every team member can reproduce them:
- **Diagnoses:** the tables' standard SNOMED concepts.
- **Labels:** rebuilt with EHRSHOT's published definitions.
- **CLMBR fingerprints:** only where someone has their own full-release copy.

This notebook checks that the conclusions don't depend on those choices:

1. **Rebuilt vs official labels.** How closely do the rebuilt labels match Stanford's? *(Needs a full-release copy; skipped otherwise.)*
2. **Coding era.** The tables also keep each diagnosis's original billing code (ICD-9 before Oct 2015, ICD-10 after). Graphs built on those codes split by era. Does era explain the homophily?
3. **Billing codes instead of concepts.** Re-run the full analysis on the original ICD codes.
4. **Official labels instead of rebuilt.** Re-run the full analysis on the same concept graphs with Stanford's labels. *(Full release only.)*

All computation goes through `ehrgraph.netstats.task_analysis`, and only aggregate results are written.

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_data_sources  # noqa: E402

sources = ensure_data_sources()

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ehrgraph.data.omop import OmopDatabase, OmopTables
from ehrgraph.data.omop_labels import ESSENTIAL_HYPERTENSION, rebuild_task_labels
from ehrgraph.eval.aggregate import write_aggregate
from ehrgraph.netstats.task_analysis import (
    OFFICIAL_LABELS_VARIANT,
    PRIMARY_VARIANT,
    SOURCE_CODES_VARIANT,
    analyze_task,
    coding_era,
    era_stats,
    task_graphs_from_labels,
)
from ehrgraph.viz import network_figures as nf

TASKS = {"guo_los": "Task A: Long Length of Stay", "new_hypertension": "Task B: Hypertension"}
results_dir = _repo_root / "results" / "aggregate"
figures_dir = results_dir / "figures"
nf.apply_style()

t0 = time.time()
tables = OmopTables(sources.trimmed)
concept_db = OmopDatabase(tables)
source_db = OmopDatabase(tables, codes="source")
rebuilt = {task: rebuild_task_labels(tables, task) for task in TASKS}
primary = pd.concat(
    [pd.read_csv(results_dir / f) for f in ("network_stats_task_a.csv", "network_stats_task_b.csv")]
)
primary = primary[primary["seed"].isna()]
rows = []
print(f"tables loaded in {time.time() - t0:.0f}s")


def ref(task, metric, value, variant=PRIMARY_VARIANT):
    rows.append(
        dict(
            task=task,
            condition="reference",
            graph_variant=variant,
            metric=metric,
            value=float(value),
        )
    )

## Part 1 — Rebuilt vs official labels (full release only)

A *prediction time* is one (patient, moment) the benchmark asks about. For each task: how many of Stanford's prediction times the rebuild also produces, and how often the two agree on the outcome where both exist.

In [ ]:
official = {}
if sources.has_full_release:
    out = []
    for task in TASKS:
        off = pd.read_csv(
            sources.assets / "benchmark" / task / "labeled_patients.csv",
            parse_dates=["prediction_time"],
        )
        official[task] = off
        rb = rebuilt[task]
        r = dict(zip(zip(rb.patient_id, rb.prediction_time, strict=True), rb.value, strict=True))
        o = dict(
            zip(
                zip(off.patient_id, off.prediction_time, strict=True),
                off.value.astype(bool),
                strict=True,
            )
        )
        shared = set(r) & set(o)
        agree = float(np.mean([bool(r[k]) == o[k] for k in shared]))
        stats = {
            "rebuilt_labels": len(r),
            "official_labels": len(o),
            "shared_prediction_times": len(shared),
            "shared_share_of_official": len(shared) / len(o),
            "outcome_agreement": agree,
            "rebuilt_prevalence": float(np.mean(list(r.values()))),
            "official_prevalence": float(np.mean(list(o.values()))),
        }
        for metric, value in stats.items():
            ref(task, metric, value)
        out.append({"task": TASKS[task]} | stats)
    display(pd.DataFrame(out).set_index("task").round(4).T)  # noqa: F821
else:
    print("No full-release copy on this machine -- Part 1 and Part 4 are skipped.")

**Why hypertension matches less closely (93% of prediction times):** EHRSHOT's definition says discharges where the patient *already* has essential hypertension are skipped. Stanford's released labels nevertheless include some already-diagnosed patients. The rebuild follows the written rule, so it leaves them out. The count below uses SNOMED 59621000 and all its sub-concepts, the definition EHRSHOT states.

In [ ]:
if sources.has_full_release:
    htn = {f"OMOP/{c}" for c in tables.descendants([ESSENTIAL_HYPERTENSION])}
    off = official["new_hypertension"]
    already = [
        any(e.code in htn and e.time < pt for e in concept_db[pid].events)
        for pid, pt in zip(off.patient_id, off.prediction_time, strict=True)
    ]
    off = off.assign(already=already)
    pos, neg = off[off.value.astype(bool)], off[~off.value.astype(bool)]
    for metric, value in (
        ("official_positives_already_diagnosed", pos.already.sum()),
        ("official_negatives_already_diagnosed", neg.already.sum()),
        ("official_positive_share_already_diagnosed", pos.already.mean()),
        ("official_negative_share_already_diagnosed", neg.already.mean()),
    ):
        ref("new_hypertension", metric, value)
    for name, grp in (("positives", pos), ("negatives", neg)):
        n, share = grp.already.sum(), grp.already.mean()
        print(f"official {name} already diagnosed: {n} of {len(grp)} ({share:.1%})")

## Part 2 and 3 — Coding era, and graphs built on billing codes

Each node gets an era from the billing codes in its safe vector:
- **icd9** or **icd10**: only one coding system appears.
- **mixed**: both appear.
- **none**: no diagnoses before the prediction time.

For each graph, three numbers:
- **Cross-era links:** links joining an ICD-9-only node to an ICD-10-only node.
- **Era assortativity:** how strongly nodes link within their own era (1 = only ever within era).
- **Era-only homophily:** shuffle outcomes *within* each era and re-measure homophily. What survives is what the era split alone could produce.

Then the whole analysis runs again on the billing-code graphs.

In [ ]:
def era_rows(task, variant, graphs, era, labels):
    for condition in ("safe", "leaky"):
        s = era_stats(graphs[condition], era, labels)
        for metric in (
            "icd9_icd10_cross_edges",
            "icd9_icd10_cross_edge_fraction",
            "era_assortativity",
        ):
            rows.append(
                dict(
                    task=task,
                    condition=condition,
                    graph_variant=variant,
                    metric=metric,
                    value=s[metric],
                )
            )
        rows.append(
            dict(
                task=task,
                condition=condition,
                graph_variant=variant,
                metric="homophily_era_only",
                value=s["homophily_era_only"],
                ci_low=s["homophily_era_only_lo"],
                ci_high=s["homophily_era_only_hi"],
            )
        )


source_runs, era_tables = {}, []
for task in TASKS:
    t0 = time.time()
    tg_src = task_graphs_from_labels(
        source_db, rebuilt[task], task, clmbr_path=sources.clmbr_features
    )
    an_src = analyze_task(tg_src, graph_variant=SOURCE_CODES_VARIANT)
    source_runs[task] = an_src
    rows.extend(an_src.rows)
    matrix, codes = tg_src.vectors["safe"]
    era = dict(zip(tg_src.nodes, coding_era(matrix, codes), strict=True))
    era_rows(task, SOURCE_CODES_VARIANT, tg_src.graphs, era, tg_src.labels)
    tg_con = task_graphs_from_labels(
        concept_db, rebuilt[task], task, n_null=1
    )  # era of the primary graphs
    era_rows(task, PRIMARY_VARIANT, tg_con.graphs, era, tg_con.labels)
    d = pd.DataFrame(
        {"era": [era[n] for n in tg_src.nodes], "y": [tg_src.labels[n] for n in tg_src.nodes]}
    )
    for e, g in d.groupby("era"):
        era_tables.append(
            {"task": TASKS[task], "era": e, "nodes": len(g), "outcome rate": round(g.y.mean(), 3)}
        )
    print(f"{TASKS[task]}: billing-code analysis + era checks in {time.time() - t0:.0f}s")
pd.DataFrame(era_tables).set_index(["task", "era"])

In [ ]:
ERA = [
    "icd9_icd10_cross_edges",
    "icd9_icd10_cross_edge_fraction",
    "era_assortativity",
    "homophily_era_only",
]
d = pd.DataFrame(rows)
d = d[d["metric"].isin(ERA)]
t = d.pivot_table(index=["task", "graph_variant", "condition"], columns="metric", values="value")
actual = pd.concat([primary, pd.DataFrame(rows)])
actual = actual[(actual["metric"] == "class_insensitive_homophily") & actual["seed"].isna()]
actual = actual.set_index(["task", "graph_variant", "condition"])["value"]
t["homophily (actual)"] = [actual.get(i, np.nan) for i in t.index]
t[ERA + ["homophily (actual)"]].round(4)

## Part 4 — Official labels on the primary graphs (full release only)

The same concept-based graphs, built for Stanford's own prediction times and labels instead of the rebuilt ones.

In [ ]:
official_runs = {}
if sources.has_full_release:
    for task in TASKS:
        t0 = time.time()
        tg_off = task_graphs_from_labels(
            concept_db, official[task], task, clmbr_path=sources.clmbr_features
        )
        official_runs[task] = analyze_task(tg_off, graph_variant=OFFICIAL_LABELS_VARIANT)
        rows.extend(official_runs[task].rows)
        print(f"{TASKS[task]}: official-label analysis in {time.time() - t0:.0f}s")

## All checks side by side

In [ ]:
KEY = {
    "class_insensitive_homophily": "homophily",
    "class_insensitive_homophily_excl_same_patient": "homophily, no same-patient links",
    "leiden_outcome_eta_squared": "outcome explained by community (η²)",
    "dirichlet_energy_clmbr": "Dirichlet energy (CLMBR)",
    "max_single_code_outcome_gap": "strongest single-code gap",
    "same_patient_edge_fraction": "same-patient share of links",
}
VARIANTS = [
    (PRIMARY_VARIANT, "primary"),
    (SOURCE_CODES_VARIANT, "billing codes"),
    (OFFICIAL_LABELS_VARIANT, "official labels"),
]
allrows = pd.concat([primary, pd.DataFrame(rows)])
allrows = allrows[allrows["seed"].isna()]


def v(task, variant, cond, metric, col="value"):
    hit = allrows[
        (allrows.task == task)
        & (allrows.graph_variant == variant)
        & (allrows.condition == cond)
        & (allrows.metric == metric)
    ]
    return float(hit[col].iloc[0]) if len(hit) else np.nan


table = []
for task in TASKS:
    for metric, name in KEY.items():
        for cond in ("safe", "leaky", "null_model"):
            entry = {"task": TASKS[task], "measure": name, "graph": cond}
            for var, label in VARIANTS:
                entry[label] = v(task, var, cond, metric)
            if not all(np.isnan(entry[label]) for _, label in VARIANTS):
                table.append(entry)
pd.DataFrame(table).set_index(["task", "measure", "graph"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharex=True)
for ax, task in zip(axes, TASKS, strict=True):
    present = [
        (var, name)
        for var, name in VARIANTS
        if not np.isnan(v(task, var, "safe", "class_insensitive_homophily"))
    ]
    for row, (var, _name) in enumerate(present):
        y0 = len(present) - 1 - row
        ax.hlines(
            y0,
            v(task, var, "null_model", "class_insensitive_homophily", "ci_low"),
            v(task, var, "null_model", "class_insensitive_homophily", "ci_high"),
            color=nf.COLORS["null"],
            lw=8,
            alpha=0.35,
        )
        for dy, cond in ((0.12, "safe"), (-0.12, "leaky")):
            a = v(task, var, cond, "class_insensitive_homophily")
            b = v(task, var, cond, "class_insensitive_homophily_excl_same_patient")
            ax.plot([a, b], [y0 + dy] * 2, color=nf.COLORS[cond], lw=1.2, alpha=0.6)
            ax.plot(a, y0 + dy, "o", color=nf.COLORS[cond], ms=8, label=cond if row == 0 else None)
            ax.plot(b, y0 + dy, "o", mfc="#fcfcfb", mec=nf.COLORS[cond], mew=2, ms=8)
    ax.set_yticks(range(len(present)), [n for _, n in present][::-1])
    ax.set_ylim(-0.6, len(present) - 0.4)
    ax.grid(axis="y", visible=False)
    ax.set_title(TASKS[task], fontsize=11)
    ax.set_xlabel("class-insensitive homophily (● all links, ○ no same-patient links)")
axes[0].legend(loc="lower right")
fig.tight_layout()
nf.save(fig, figures_dir / "11_robustness_homophily.png")
plt.show()

In [ ]:
out = write_aggregate(rows, results_dir / "robustness_checks.csv")
saved = pd.read_csv(out)
print(f"wrote {len(saved)} rows -> {out.relative_to(_repo_root)}")
saved.groupby(["task", "graph_variant"])["metric"].count()

## Findings

**Every Project 1 conclusion holds on the course tables, and under each alternative.** The size of the Task B leakage effect depends on construction choices, and the report should say so.

**1. The rebuilt labels match Stanford's closely.**

| | Long stay | Hypertension |
|---|---|---|
| Stanford's prediction times also in the rebuild | 6,930 of 6,995 (99.1%) | 3,499 of 3,764 (93.0%) |
| Outcome agreement where both exist | **99.84%** | **99.40%** |
| Prevalence, rebuilt vs official | 0.254 vs 0.253 | 0.128 vs 0.137 |

The hypertension gap is mostly one rule. EHRSHOT's definition skips discharges where the patient already has essential hypertension, yet **55 of Stanford's 516 positives (10.7%) and 84 of 3,248 negatives (2.6%)** already carry the diagnosis (SNOMED 59621000 or a sub-concept) before the prediction time. The rebuild follows the written rule.

**2. Coding era splits the billing-code graphs, but it doesn't explain homophily.** On billing codes, only **1 of 18,693** long-stay links and **5 of 9,892** hypertension links join an ICD-9-only node to an ICD-10-only node (era assortativity 0.72 in both). The primary concept graphs open that wall to 3–4% cross-era links. Shuffling outcomes within each era leaves homophily of **0.004** (long stay) and **0.014** (hypertension), against actual 0.211 and 0.226: era explains at most 2% and 6%. Hypertension rates do differ by era (16.4% for ICD-9-only nodes vs 6.6% for ICD-10-only ones), but the signal lives within eras.

**3. The conclusions survive every variant.**

| Homophily, safe → leaky | Primary (concepts, rebuilt labels) | Billing codes | Official labels |
|---|---|---|---|
| Long stay | 0.211 → 0.201 | 0.209 → 0.205 | 0.210 → 0.199 |
| Hypertension | 0.226 → **0.263** (+0.038) | 0.226 → **0.282** (+0.056) | 0.279 → **0.331** (+0.051) |
| Hypertension, no same-patient links | 0.115 → **0.154** (+34%) | 0.113 → **0.178** (+58%) | 0.131 → **0.191** (+46%) |
| Null homophily (all tasks, variants) | ≤ 0.010 | ≤ 0.012 | ≤ 0.011 |

- **Long stay:** no leakage effect in any version.
- **Hypertension:** a leakage effect in every version, from +0.038 to +0.056.
- **CLMBR alignment:** Dirichlet energy is 0.65–0.69 everywhere, against 0.95 for random graphs.

**4. Stronger answer key, weaker effect.** Under concept codes the leaked diagnosis is a near-perfect separator: present for 99.6% of positives, a single-code gap of 0.87, against 0.64 on billing codes. Yet its effect on homophily is *smaller* (+0.038 vs +0.056), because one merged, very common concept is heavily down-weighted by TF-IDF. How much leakage reaches a graph depends on how the graph is built.

**5. Official labels make the honest graph look better than it is.** With Stanford's labels, safe Task B homophily is 0.279, against 0.226 with rebuilt ones. Much of that difference comes from the already-diagnosed patients, whose existing diagnosis is legitimately in their history.

### What the report can say

1. Honest patient graphs are organized by outcome, far beyond chance: in every version, on the data every team member has.
2. Leakage is inert for long stay and real for hypertension. That holds in every version, with a gap of +0.038 to +0.056 depending on construction.
3. Code harmonization, same-patient links and cohort definition each measurably change how much leakage a graph carries. That is the concrete case for the Project 2 graph-variant sweep.

### Caveats

- Rebuilt labels are not Stanford's labels: 99.1% / 93% coverage, 99.8% / 99.4% agreement. Benchmark-comparable numbers (Project 2's baseline) still need the official labels and the full release.
- Why Stanford's exclusion missed the already-diagnosed patients is not established. It's a question for Stanford's team, not a claim.

### Done when
- [x] #35 and #37: rebuilt-vs-official labels, coding era, billing-code graphs and official-label graphs, all from aggregate output (`results/aggregate/robustness_checks.csv`, `11_robustness_homophily.png`)